In [0]:
dbutils.widgets.text("catalog","sky_dev")
catalog = dbutils.widgets.get("catalog")


from pyspark.sql import functions as F

In [0]:
def sid(col):
    std_col = F.trim(F.upper(F.col(col)))
    return  F.when(std_col == "", None).otherwise(std_col)

### Clean Passengers

In [0]:
regions = spark.read.table(f'{catalog}.bronze.regions')   

In [0]:
regions_std = regions.select(
    "r_regionkey",
    F.trim(F.initcap(F.col("r_name"))).alias("r_name"),
    F.trim(F.col("r_comment")).alias("r_comment")
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("r_regionkey").isNotNull()

In [0]:
regions_valid = regions_std.filter(valid_primary_key)
regions_valid.write.mode("overwrite").saveAsTable(f"{catalog}.silver.regions")

In [0]:
quarantine_regions = regions_std.withColumn("reject_reason", F.when(~valid_primary_key, "INVALID_PRIMARY_KEY")) \
    .select(F.to_csv(F.struct("*")).alias("raw_record"), "reject_reason") \
        .filter(F.col("reject_reason").isNotNull())
quarantine_regions.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_regions")

### Clean Nations

In [0]:
nations = spark.read.table(f"{catalog}.bronze.nations")
regions = spark.read.table(f"{catalog}.silver.regions")

In [0]:
nations_std = nations.select(
    "n_nationkey",
    F.trim(F.upper(F.col("n_name"))).alias("n_name"),
    "n_regionkey",
    F.trim(F.col("n_comment")).alias("n_comment")
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("n_nationkey").isNotNull()
referential_integrity = F.col("n_regionkey").isNotNull() & F.col("n_regionkey").isin([row.r_regionkey for row in regions.select("r_regionkey").distinct().collect()])

In [0]:
valid_nations = nations_std.filter(valid_primary_key & referential_integrity)
valid_nations.write.mode("overwrite").saveAsTable(f"{catalog}.silver.nations")

In [0]:
reject_reasons = F.array_remove(
    F.array(
        F.when(~valid_primary_key, "INVALID_PRIMARY_KEY"),
        F.when(~referential_integrity, "VIOLATES_REFERENTIAL_INTEGRITY")
    ),
    None
)

quarantine_nations = nations_std.withColumn("reject_reasons", reject_reasons) \
                    .filter(F.size(F.col("reject_reasons")) > 0) \
                    .select(
                        F.to_csv(F.struct("*")).alias("raw_record"),
                        "reject_reasons"
                    )
quarantine_nations.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_nations")

### clean passangers

In [0]:
passengers = spark.read.table(f"{catalog}.bronze.passengers")
nations = spark.read.table(f"{catalog}.silver.nations")

In [0]:
passengers_std = passengers.select(
    "pa_passengerid",
    F.trim(F.initcap(F.col("pa_name"))).alias("pa_name"),
    F.trim(F.col("pa_address")).alias("pa_address"),
    "pa_nationkey",
    F.trim(F.col("pa_phone")).alias("pa_phone"),
    "pa_loyaltypoints",
    F.trim(F.upper(F.col("pa_travelpurpose"))).alias("pa_travelpurpose"),
    F.trim(F.col("pa_comment")).alias("pa_comment")
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("pa_passengerid").isNotNull()
referential_integrity = F.col("pa_nationkey").isNotNull() & F.col("pa_nationkey").isin([row.n_nationkey for row in nations.select("n_nationkey").distinct().collect()])

In [0]:
valid_passengers = passengers_std.filter(valid_primary_key & referential_integrity)
valid_passengers.write.mode("overwrite").saveAsTable(f"{catalog}.silver.passengers")

In [0]:
%sql
select * from sky_dev.silver.quarantine_passengers

In [0]:
reject_reasons = F.array_compact(
  F.array(
    F.when(~valid_primary_key, "INVALID_PRIMARY_KEY"),
    F.when(~referential_integrity, "VIOLATES_REFERENTIAL_INTEGRITY")
  )
)
quarantine_passengers = passengers_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0) \
    .select(F.to_csv(F.struct("*")).alias("raw_record"), "reject_reasons")
quarantine_passengers.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_passengers")

### Clean Airline

In [0]:
airlines = spark.read.table(f"{catalog}.bronze.airlines")

In [0]:
airlines_std = airlines.select(
    "al_airlineid",
    F.trim(F.initcap(F.col("al_name"))).alias("al_name"),
    F.trim(F.col("al_address")).alias("al_address"),
    "al_nationkey",
    "al_rating",
    F.trim(F.col("al_comment")).alias("al_comment")
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("al_airlineid").isNotNull()
referential_integrity = F.col("al_nationkey").isNotNull() & F.col("al_nationkey").isin([row.n_nationkey for row in nations.select("n_nationkey").distinct().collect()])

In [0]:
valid_airlines = airlines_std.filter(valid_primary_key & referential_integrity)
valid_airlines.write.mode("overwrite").saveAsTable(f"{catalog}.silver.airlines")

In [0]:
reject_reasons = F.array_compact(
  F.array(
    F.when(~valid_primary_key, "INVALID_PRIMARY_KEY"),
    F.when(~referential_integrity, "VIOLATES_REFERENTIAL_INTEGRITY")
  )
)
quarantine_airlines = airlines_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0) \
    .select(F.to_csv(F.struct("*")).alias("raw_record"), "reject_reasons")
quarantine_airlines.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_airline")

### Clean Bookings

In [0]:
bookings = spark.read.table(f"{catalog}.bronze.bookings")
passengers = spark.read.table(f"{catalog}.silver.passengers")

In [0]:
bookings_std = bookings.select(
    "bk_bookingid",
    "bk_passengerid",
    F.trim(F.upper(F.col("bk_status"))).alias("bk_status"),
    "bk_totalprice",
    F.to_date(F.col("bk_bookingdate"), "yyyy-MM-dd").alias("bk_bookingdate"),
    F.to_date(F.col("bk_flightdate"), "yyyy-MM-dd").alias("bk_flightdate"),
    F.trim(F.upper(F.col("bk_bookingpriority"))).alias("bk_bookingpriority"),
    F.trim(F.initcap(F.col("bk_agent"))).alias("bk_agent"),
    F.trim(F.col("bk_comment")).alias("bk_comment")
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("bk_bookingid").isNotNull()
referential_integrity = F.col("bk_passengerid").isNotNull() & F.col("bk_passengerid").isin([row.pa_passengerid for row in passengers.select("pa_passengerid").distinct().collect()])
valid_booking_date = (F.col("bk_bookingdate") >= F.to_date(F.lit('1994-01-01'))) & (F.col("bk_bookingdate") <= F.to_date(F.lit('1998-12-31')))
valid_flight_date = (F.col("bk_flightdate") >= F.to_date(F.lit('1994-01-01'))) & (F.col("bk_flightdate") <= F.to_date(F.lit('1998-12-31')))

In [0]:
valid_bookings = bookings_std.filter(valid_primary_key & referential_integrity & valid_booking_date & valid_flight_date)
valid_bookings.write.mode("overwrite").saveAsTable(f"{catalog}.silver.bookings")

In [0]:
reject_reasons = F.array_compact(
  F.array(
    F.when(~valid_primary_key, "INVALID_PRIMARY_KEY"),
    F.when(~referential_integrity, "VIOLATES_REFERENTIAL_INTEGRITY"),
    F.when(~valid_booking_date, "INVALID_BOOKING_DATE"),
    F.when(~valid_flight_date, "INVALID_FLIGHT_DATE")
  )
)
quarantine_bookings = bookings_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0) \
    .select(F.to_csv(F.struct("*")).alias("raw_record"), "reject_reasons")
quarantine_bookings.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_bookings")

### Clean routes

In [0]:
routes = spark.read.table(f"{catalog}.bronze.routes")

In [0]:
routes.display()

In [0]:
routes_std = routes.select(
    "ro_routeid",
    F.trim(F.upper(F.col("ro_name"))).alias("ro_name"),
    F.trim(F.initcap(F.col("ro_aircraft"))).alias("ro_aircraft"),
    F.trim(F.initcap(F.col("ro_airline_brand"))).alias("ro_airline_brand"),
    F.trim(F.upper(F.col("ro_routetype"))).alias("ro_routetype"),
    "ro_capacity",
    F.trim(F.upper(F.col("ro_cabinconfig"))).alias("ro_cabinconfig"),
    "ro_baseprice",
    F.trim(F.col("ro_comment")).alias("ro_comment")
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("ro_routeid").isNotNull()

valid_routes = routes_std.filter(valid_primary_key)
valid_routes.write.mode("overwrite").saveAsTable(f"{catalog}.silver.routes")

In [0]:
quarantine_routes = routes_std.withColumn("reject_reasons", F.when(~valid_primary_key, "INVALID_PRIMARY_KEY")) \
    .filter(F.col("reject_reasons").isNotNull()) \
        .select(F.to_csv(F.struct("*")).alias("raw_record"), "reject_reasons")
quarantine_routes.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_routes")

In [0]:
%sql
select * from sky_dev.silver.quarantine_routes

### Clean Airline routes

In [0]:
a_routes = spark.read.table(f"{catalog}.bronze.airline_routes")
airlines = spark.read.table(f"{catalog}.silver.airlines")
routes = spark.read.table(f"{catalog}.silver.routes")


In [0]:
a_routes.display()

In [0]:
a_routes_std = a_routes.select(
    "ar_airlineid",
    "ar_routeid",
    "ar_available_seats",
    "ar_operatingcost",
    "ar_comment"
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("ar_airlineid").isNotNull() & F.col("ar_routeid").isNotNull()
referential_integrity = (F.col("ar_airlineid").isNotNull() & F.col("ar_airlineid").isin([row.al_airlineid for row in airlines.select("al_airlineid").distinct().collect()])) & \
(F.col("ar_routeid").isNotNull() & F.col("ar_routeid").isin([row.ro_routeid for row in routes.select("ro_routeid").distinct().collect()]))
valid_cost = F.col("ar_operatingcost") > 0

In [0]:
valid_a_routes = a_routes_std.filter(valid_primary_key & valid_cost & referential_integrity)
valid_a_routes.write.mode("overwrite").saveAsTable(f"{catalog}.silver.airline_routes")

In [0]:
reject_reasons = F.array_compact(
  F.array(
    F.when(~valid_primary_key, "INVALID_PRIMARY_KEY"),
    F.when(~valid_cost, "INVALID_COST"),
    F.when(~referential_integrity, "VOILATES_REFERENTIAL_INTEGRITY")
  )
)
quarantine_a_routes = a_routes_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0) \
    .select(F.to_csv(F.struct("*")).alias("raw_record"), "reject_reasons")
quarantine_a_routes.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_airline_routes")

### Clean flight bookings

In [0]:
f_bookings = spark.read.table(f"{catalog}.bronze.flight_bookings")
bookings = spark.read.table(f"{catalog}.silver.bookings")
routes = spark.read.table(f"{catalog}.silver.routes")
airlines = spark.read.table(f"{catalog}.silver.airlines")


In [0]:
f_bookings.display()

In [0]:
f_bookings_std = f_bookings.select(
    "fb_bookingid",
    "fb_routeid",
    "fb_airlineid",
    "fb_linenumber",
    "fb_seats",
    "fb_ticketprice",
    "fb_discount",
    "fb_tax",
    "fb_refundflag",
    "fb_segmentstatus",
    "fb_departuredate",
    "fb_commitdate",
    "fb_arrivaldate",
    F.trim(F.upper(F.col("fb_cabinclass"))).alias("fb_cabinclass"),
    F.trim(F.col("fb_comment")).alias("fb_comment")
).dropDuplicates()

In [0]:
# validations
valid_primary_key = F.col("fb_bookingid").isNotNull() & F.col("fb_linenumber").isNotNull()
referential_integrity = (F.col("fb_bookingid").isNotNull() & F.col("fb_bookingid").isin([row.bk_bookingid for row in bookings.select("bk_bookingid").distinct().collect()])) & \
(F.col("fb_routeid").isNotNull() & F.col("fb_routeid").isin([row.ro_routeid for row in routes.select("ro_routeid").distinct().collect()])) & \
(F.col("fb_airlineid").isNotNull() & F.col("fb_airlineid").isin([row.al_airlineid for row in airlines.select("al_airlineid").distinct().collect()]))
valid_ticket_price = F.col("fb_ticketprice") > 0
valid_discount = (F.col("fb_discount") >= 0) & (F.col("fb_discount") <= 1)
valid_seat = F.col("fb_seats") >= 1

booking_dates = {row.bk_bookingid: row.bk_bookingdate for row in bookings.select("bk_bookingid", "bk_bookingdate").distinct().collect()}
booking_date_map = F.create_map([F.lit(x) for kv in booking_dates.items() for x in kv])

valid_departure_date = (F.col("fb_departuredate") >= F.to_date(F.lit('1994-01-01'))) & (F.col("fb_departuredate") <= F.to_date(F.lit('1998-12-31'))) & \
(F.col("fb_departuredate") >= booking_date_map.getItem(F.col("fb_bookingid")))

valid_arrival_date = (F.col("fb_arrivaldate") >= F.to_date(F.lit('1994-01-01'))) & (F.col("fb_arrivaldate") <= F.to_date(F.lit('1998-12-31'))) & (F.col("fb_arrivaldate") >= F.col("fb_departuredate"))

valid_commit_date = (F.col("fb_commitdate") >= F.to_date(F.lit('1994-01-01'))) & (F.col("fb_commitdate") <= F.to_date(F.lit('1998-12-31')))

valid_tax = (F.col("fb_tax") >= 0) & (F.col("fb_tax") <= 1)
# commit date, tax, 


In [0]:
valid_f_booking = f_bookings_std.filter(valid_primary_key & referential_integrity & valid_ticket_price & valid_discount & valid_seat & valid_departure_date & valid_arrival_date & valid_tax & valid_commit_date)
valid_f_booking.write.mode("overwrite").saveAsTable(f"{catalog}.silver.flight_bookings")

In [0]:
reject_reasons = F.array_compact(
  F.array(
    F.when(~valid_primary_key, "INVALID_PRIMARY_KEY"),
    F.when(~referential_integrity, "VIOLATES_REFERENTIAL_INTEGRITY"),
    F.when(~valid_ticket_price, "INVALID_TICKET_PRICE"),
    F.when(~valid_discount,"INVALID_DISCOUNT"),
    F.when(~valid_seat,"INVALID_SEATS"),
    F.when(~valid_departure_date,"INVALID_DEPARTURE_DATE"),
    F.when(~valid_arrival_date, "INVALID_ARRIVAL_DATE"),
    F.when(~valid_commit_date, "INVALID_COMMIT_DATE"),
    F.when(~valid_tax, "INVALID_TAX")
  )
)
quarantine_f_bookings = f_bookings_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0) \
    .select(F.to_csv(F.struct("*")).alias("raw_record"), "reject_reasons")
quarantine_f_bookings.write.mode("overwrite").saveAsTable(f"{catalog}.silver.quarantine_flight_bookings")

In [0]:
%sql
select count(*) from sky_dev.silver.flight_bookings

In [0]:
%sql
select * from sky_dev.silver.flight_bookings